In [ ]:
import numpy as np
from itertools import product
from multiprocessing import Pool, cpu_count
import time

# n_user = 5
for n_user in np.arange(3, 4, 1):
    h_bad = 0.1
    h_good = 1.0
    pr_h_bad = 0.5
    pr_h_good = 1.0 - pr_h_bad
    r_max = 1
    rho = np.arange(0, r_max + 1)
    P_bar_U = np.full(n_user, 2.0)
    alpha = 0.05
    max_iter = 50000
    pkt_prob = 0.9

    h_values = [h_bad, h_good]
    h = np.array( list(product(h_values, repeat=n_user)), dtype=np.float64)
    num_h = len(h)
    prob_values = [pr_h_bad, pr_h_good]
    hp = np.array( [ np.prod(p) for p in product(prob_values, repeat=n_user) ],
                    dtype=np.float64)
    W = np.array( list(product(rho, repeat=n_user)), dtype=np.float64)
    #######################
    # W = [np.zeros(n_user, dtype=int)]
    # for i in range(n_user):
    #     vec = np.zeros(n_user, dtype=int)
    #     vec[i] = 1
    #     W.append(vec)
    # for i in range(n_user):
    #     vec = np.zeros(n_user, dtype=int)
    #     vec[i] = 2
    #     W.append(vec)
    # W = np.array(W)
    #############################
    num_W = len(W)
    psi = (W > 0).astype(np.float64)

    def cal_pow_all(W, beta, h_val):
        M = len(beta)
        order = np.argsort(beta / h_val)
        W_ordered = W[:, order]
        rev_cumsum = np.cumsum( W_ordered[:, ::-1], axis=1 )[:, ::-1]
        P = np.zeros_like(W)
        for i in range(M):
            ui = order[i]
            sum_i_M = rev_cumsum[:, i]
            if i == M - 1:
                sum_ip1_M = 0
            else:
                sum_ip1_M = rev_cumsum[:, i + 1]
            P[:, ui] = ( 2.0 ** sum_i_M  - 2.0 ** sum_ip1_M  ) / h_val[ui]
        return P

    def process_channel_k(args):
        k, lambda_k, v_k = args
        h_k = h[k]
        Pk = cal_pow_all(W, lambda_k, h_k )

        # costs = hp[k] * np.sum(lambda_k * Pk - v_k * psi, axis=1 )

        costs = hp[k] * np.sum(pkt_prob* lambda_k * Pk - v_k * psi - lambda_k * P_bar_U * psi * (pkt_prob-1) , axis=1 )

        min_cost = np.min(costs)
        idx = np.where(np.abs(costs - min_cost) < 1e-10 )[0]
        mu = np.zeros(num_W)
        mu[idx] = 1.0 / len(idx)
        p_k = hp[k] * np.sum( mu[:, None] * psi, axis=0 )
        Pu_k = hp[k] * np.sum( mu[:, None] * Pk, axis=0 )
        return p_k, Pu_k

    lambda_k = np.full(n_user, 0.1)
    v_k = np.full(n_user, 0.1)

    acc_p = np.zeros(n_user)
    acc_Pu = np.zeros(n_user)
    acc_count = 0

    n_workers = cpu_count()
    print(f"Using {n_workers} workers")
    t1 = time.time()
    with Pool(processes=n_workers) as pool:
        for itr in range(max_iter):
            args = [( k,lambda_k.copy(), v_k.copy()) for k in range(num_h)]
            results = pool.map( process_channel_k, args )

            p = np.zeros(n_user)
            P_u = np.zeros(n_user)

            for p_k, Pu_k in results:
                p += p_k
                P_u += Pu_k

            if itr > 2:
                acc_p += p
                acc_Pu += P_u
                acc_count += 1
            eta = alpha / np.sqrt(itr + 1)

            # lambda_k = np.maximum(  0.0,  lambda_k + eta * ( P_u - P_bar_U ) )
            
            lambda_k = np.maximum(  0.0,  lambda_k + eta * ( pkt_prob * P_u - P_bar_U*(pkt_prob- pkt_prob* p + p) ) )

            v_k = np.maximum(1e-10, v_k + eta * (1.0 / np.sqrt(v_k) - p ) )
            if itr % 10000 == 0:
                print(f"Iteration = {itr}")
    runtime = time.time() - t1
    p_er = acc_p / acc_count
    weights  = np.ones(n_user) / n_user
    pkt_prob_arr = pkt_prob * np.ones(n_user)
    age_bar = np.sum(weights * pkt_prob_arr * (1.0 / p_er - 1.0))
    print('M:', n_user)
    print("Final average age:", np.round(age_bar, 6))
    print()
    print("Runtime:")
    print(np.round(runtime, 3), "seconds")
    print("==================================================")
    print('beta', lambda_k)

In [ ]:
np.round(erg_mu, 3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

M = np.arange(1, 10)
M10 = np.arange(1, 11)
M_greedy = np.arange(1, 8)

va_srp_noma = [11.2, 13.2, 19.1, 31.2, 60.6,
               111.4, 236.2, 525.22, 1834.9]

va_srp_tdma = [2.4, 3.02, 4.27, 6.451, 11.92,
               21.61, 43.27, 82.88, 150.89]

greedy_online = [0.00171, 0.00401, 0.00965,
                 0.02694, 0.07531, 0.2193, 0.7012]

max_vaoi_first = [0.001511, 0.002427, 0.003802,
                  0.005247, 0.00706, 0.009152,
                  0.011354, 0.013941, 0.016317]

va_srp_noma_online = [1.1050662994384766e-05,
                      1.0249600410461425e-05,
                      1.0287387371063232e-05,
                      1.05422043800354e-05,
                      1.159574031829834e-05,
                      1.4549059867858888e-05,
                      2.1833624839782714e-05,
                      4.3780875205993654e-05,
                      1.0800664186477662e-04]

va_srp_tdma_online = [1.1333439350128174e-05,
                      1.0347230434417725e-05,
                      1.0163378715515137e-05,
                      1.0159239768981933e-05,
                      1.0177352428436279e-05,
                      1.0281598567962647e-05,
                      1.0280575752258301e-05,
                      1.0331354141235351e-05,
                      1.0438566207885742e-05]

fig, ax1 = plt.subplots(figsize=(7.2, 5))

ax2 = ax1.twinx()

l1 = ax1.plot(M, va_srp_noma,
              marker='o',
              linewidth=2,
              label='VA-SRP NOMA Offline')

l2 = ax1.plot(M, va_srp_tdma,
              marker='s',
              linewidth=2,
              label='VA-SRP TDMA Offline')

l3 = ax2.plot(M_greedy, greedy_online,
              marker='d',
              linestyle='--',
              linewidth=2,
              label='Greedy Online')

l4 = ax2.plot(M, max_vaoi_first,
              marker='^',
              linestyle='-.',
              linewidth=2,
              label='Max-VAoI-First Online')

l5 = ax2.plot(M, va_srp_noma_online,
              marker='x',
              linestyle=':',
              linewidth=2,
              label='VA-SRP NOMA Online')

l6 = ax2.plot(M, va_srp_tdma_online,
              marker='P',
              linestyle=':',
              linewidth=2,
              label='VA-SRP TDMA Online')

ax1.set_yscale('log')
ax2.set_yscale('log')

ax1.set_xlabel('Number of users $M$')

ax1.set_ylabel('Offline runtime (sec)')
ax2.set_ylabel('Online runtime per slot (sec)')

ax1.grid(True, which='both', linestyle='--')

lines = l1 + l2 + l3 + l4 + l5 + l6
labels = [line.get_label() for line in lines]

ax1.legend(lines, labels,
           loc='upper left',
           fontsize=8)

plt.tight_layout()
plt.show()

# Greedy NOMA

In [ ]:
import numpy as np
import cvxpy as cp
import time
from itertools import product

def feasible_F_rhos(bits, h, P_avail):
    M = len(bits)
    # Power variables
    P = cp.Variable(M, nonneg=True)
    constraints = [
        P >= 0,
        P <= P_avail    ]
    for mask in range(1, 2**M):
        subset_idx = [i for i in range(M) if (mask >> i) & 1]
        lhs = cp.sum([h[i] * P[i] for i in subset_idx])
        rhs = 2**(np.sum(bits[subset_idx])) - 1
        constraints.append(lhs >= rhs)
    # Objective: minimum total power
    problem = cp.Problem(cp.Minimize(cp.sum(P)), constraints)
    try:
        problem.solve(solver=cp.SCS, verbose=False)
        if P.value is None:
            return np.inf, np.zeros(M)
        return problem.value, np.maximum(P.value, 0)
    except:
        return np.inf, np.zeros(M)
pkt_prob = 0.5
M = 3                  # number of users
P_bar = 2
r_max = 2
T = 2001
rho = np.arange(0, r_max + 1, 1)
W = np.array(list(product(rho, repeat=M)))
P_consume = np.zeros(M)
age = np.zeros(M, dtype=np.int64)
sum_age = np.zeros(M, dtype=np.int64)
sum_time = 0
for t in range(1, T + 1):
    # Channel states
    h_vals = np.random.choice(
        [1.0, 0.1],
        size=M,
        p=[0.5, 0.5]    )
    # Packet arrivals
    pkts = np.random.choice(
        [1, 0],
        size=M,
        p=[pkt_prob, 1 - pkt_prob]    )
    t_start = time.time()
    # Remaining battery/energy
    Pow = np.array([
        t * P_bar - P_consume[i]
        for i in range(M)    ])
    # Age udate
    age += pkts
    # Queue state
    q_state = (age > 0).astype(int)
    feasible_list = []
    for rhos in W:
        bits = np.array(rhos) * q_state
        res2, Ps = feasible_F_rhos(bits, h_vals, Pow)
        if np.isfinite(res2):
            feasible_list.append(bits)
    if len(feasible_list) > 0:
        # Score = resulting total age
        scores = [
            np.sum(age * np.where(x > 0, 0, 1))
            for x in feasible_list        ]
        min_score = min(scores)
        best_bits_age = [
            x for x, s in zip(feasible_list, scores)
            if s == min_score        ]
        # Tie-breaker: maximize transmissions
        best_bits = max(
            best_bits_age,
            key=lambda x: np.sum(x)        )
    else:
        best_bits = np.zeros(M)
    age = age * np.where(best_bits > 0, 0, 1)
    _, used_pow = feasible_F_rhos(
        best_bits,
        h_vals,
        Pow    )
    P_consume += used_pow
    sum_age += age
    sum_time += (time.time() - t_start)
    if t % 100 == 0:
        avg_age_per_user = sum_age / t
        print(
            f"M={M}, t={t}, "
            f"Avg age per user = {np.round(avg_age_per_user, 4)}, "
            f"Network Avg = {np.round(np.mean(avg_age_per_user), 4)}"        )

        print(
            "Average compute time per slot:",
            np.round(sum_time / t, 5)        )

# Greedy - TDMA

In [ ]:
import numpy as np
import cvxpy as cp
import time

def feasible_F_rhos_tdma(bits, h, P_avail):
    M = len(bits)
    P = cp.Variable(M, nonneg=True)
    constraints = [
        P >= 0,
        P <= P_avail    ]
    active_users = np.where(bits > 0)[0]
    for i in active_users:
        constraints.append(
            h[i] * P[i] >= 2**bits[i] - 1        )
    problem = cp.Problem(
        cp.Minimize(cp.sum(P)),
        constraints    )
    try:
        problem.solve(solver=cp.SCS, verbose=False)
        if P.value is None:
            return np.inf, np.zeros(M)
        return problem.value, np.maximum(P.value, 0)
    except:
        return np.inf, np.zeros(M)

for M in range(7, 11, 1):
    pkt_prob = 0.5
    P_bar = 2
    T = 2001
    W = [np.zeros(M, dtype=int)]
    for i in range(M):
        vec = np.zeros(M, dtype=int)
        vec[i] = 1
        W.append(vec)
    for i in range(M):
        vec = np.zeros(M, dtype=int)
        vec[i] = 2
        W.append(vec)
    W = np.array(W)
    P_consume = np.zeros(M)
    age = np.zeros(M, dtype=np.int64)
    sum_age = np.zeros(M, dtype=np.int64)
    sum_time = 0
    for t in range(1, T + 1):
        h_vals = np.random.choice(
            [1.0, 0.1],
            size=M,
            p=[0.5, 0.5]    )
        pkts = np.random.choice(
            [1, 0],
            size=M,
            p=[pkt_prob, 1 - pkt_prob]    )
        t_start = time.time()
        Pow = np.array([
            t * P_bar - P_consume[i]
            for i in range(M)    ])
        age += pkts
        q_state = (age > 0).astype(int)
        feasible_list = []
        for rhos in W:
            bits = np.array(rhos) * q_state
            res2, Ps = feasible_F_rhos_tdma(
                bits,
                h_vals,
                Pow        )
            if np.isfinite(res2):
                feasible_list.append(bits)
        if len(feasible_list) > 0:
            scores = [
                np.sum(age * np.where(x > 0, 0, 1))
                for x in feasible_list        ]
            min_score = min(scores)
            best_bits_age = [
                x for x, s in zip(feasible_list, scores)
                if s == min_score        ]
            best_bits = max(
                best_bits_age,
                key=lambda x: np.sum(x)        )
        else:
            best_bits = np.zeros(M)
        age = age * np.where(best_bits > 0, 0, 1)
        _, used_pow = feasible_F_rhos_tdma(
            best_bits,
            h_vals,
            Pow    )
        P_consume += used_pow
        sum_age += age
        sum_time += (time.time() - t_start)
        if t % 500 == 0:
            avg_age_per_user = sum_age / t
            print(
                f"TDMA | M={M}, t={t}, "
                f"Avg age per user = {np.round(avg_age_per_user, 4)}, "
                f"Network Avg = {np.round(np.mean(avg_age_per_user), 4)}"        )
            print(
                "Average compute time per slot:",
                np.round(sum_time / t, 6)        )
    print('------------------------')

In [ ]:
import numpy as np
import cvxpy as cp
import time

def build_tdma_solver(M):

    P = cp.Variable(M, nonneg=True)

    rhs_param = cp.Parameter(M, nonneg=True)

    h_param = cp.Parameter(M, nonneg=True)

    pow_param = cp.Parameter(M, nonneg=True)

    constraints = [
        P >= 0,
        P <= pow_param
    ]

    for i in range(M):

        constraints.append(
            h_param[i] * P[i] >= rhs_param[i]
        )

    problem = cp.Problem(
        cp.Minimize(cp.sum(P)),
        constraints
    )

    return (
        P,
        rhs_param,
        h_param,
        pow_param,
        problem
    )

def feasible_F_rhos_tdma(
    bits,
    h,
    P_avail,
    P,
    rhs_param,
    h_param,
    pow_param,
    problem
):

    rhs_vals = (
        2 ** bits - 1
    ).astype(np.float64)

    req_pow = (
        rhs_vals
        / np.maximum(h, 1e-12)
    )

    if np.any(req_pow > P_avail):

        return np.inf, None

    rhs_param.value = rhs_vals

    h_param.value = h

    pow_param.value = P_avail

    try:

        problem.solve(
            solver=cp.SCS,
            warm_start=True,
            verbose=False
        )

        if P.value is None:

            return np.inf, None

        return (
            problem.value,
            np.maximum(P.value, 0)
        )

    except:

        return np.inf, None

for M in range(1, 11):

    pkt_prob = 0.5

    P_bar = 2

    T = 5001

    W = [np.zeros(M, dtype=np.int8)]

    for i in range(M):

        vec1 = np.zeros(M, dtype=np.int8)
        vec1[i] = 1
        W.append(vec1)

        vec2 = np.zeros(M, dtype=np.int8)
        vec2[i] = 2
        W.append(vec2)

    W = np.array(W, dtype=np.int8)

    (
        P,
        rhs_param,
        h_param,
        pow_param,
        problem
    ) = build_tdma_solver(M)

    P_consume = np.zeros(M)

    age = np.zeros(M, dtype=np.int32)

    sum_age = np.zeros(M, dtype=np.float64)

    sum_time = 0

    for t in range(1, T + 1):

        t_start = time.time()

        h_vals = np.random.choice(
            [1.0, 0.1],
            size=M,
            p=[0.5, 0.5]
        )

        pkts = np.random.choice(
            [1, 0],
            size=M,
            p=[pkt_prob, 1 - pkt_prob]
        )

        Pow = (
            t * P_bar - P_consume
        )

        age += pkts

        q_state = (
            age > 0
        ).astype(np.int8)

        feasible_list = []

        scores_list = []

        for rhos in W:

            bits = rhos * q_state

            if np.all(bits == 0):

                feasible_list.append(
                    bits.copy()
                )

                scores_list.append(
                    np.sum(age)
                )

                continue

            res2, _ = feasible_F_rhos_tdma(
                bits,
                h_vals,
                Pow,
                P,
                rhs_param,
                h_param,
                pow_param,
                problem
            )

            if np.isfinite(res2):

                feasible_list.append(
                    bits.copy()
                )

                score = np.sum(
                    age * (bits == 0)
                )

                scores_list.append(
                    score
                )

        if len(feasible_list) > 0:

            scores_array = np.array(
                scores_list
            )

            min_score = np.min(
                scores_array
            )

            candidate_idx = np.where(
                scores_array == min_score
            )[0]

            candidate_actions = [
                feasible_list[i]
                for i in candidate_idx
            ]

            best_bits = max(
                candidate_actions,
                key=lambda x: np.sum(x)
            )

        else:

            best_bits = np.zeros(
                M,
                dtype=np.int8
            )

        age = age * (
            best_bits == 0
        )

        _, used_pow = feasible_F_rhos_tdma(
            best_bits,
            h_vals,
            Pow,
            P,
            rhs_param,
            h_param,
            pow_param,
            problem
        )

        if used_pow is not None:

            P_consume += used_pow

        sum_age += age

        sum_time += (
            time.time() - t_start
        )

        if t % 5000 == 0:

            avg_age_per_user = (
                sum_age / t
            )

            print(
                f"TDMA | M={M}, t={t}, "
                f"Avg age per user = "
                f"{np.round(avg_age_per_user, 4)}, "
                f"Network Avg = "
                f"{np.round(np.mean(avg_age_per_user), 4)}"
            )

            print(
                "Average compute time per slot:",
                np.round(sum_time / t, 6)
            )

    print('------------------------')